In [1]:
source ../modules/setup.tcl

set year 2021
set day 14

aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::display "text/markdown" "## Input\n```\n[string range $input 0 20]...\n```";

(cached)

--- Day 14: Extended Polymerization --- The incredible pressures at this depth are starting to put a strain on your submarine. The submarine has polymerization equipment that would produce suitable materials to reinforce the submarine, and the nearby volcanically-active caves should even have the necessary input elements in sufficient quantities. 
 The submarine manual contains instructions for finding the optimal polymer formula; specifically, it offers a polymer template and a list of pair insertion rules (your puzzle input). You just need to work out what polymer would result after repeating the pair insertion process a few times. 
 For example: 
 NNCB

CH -> B
HH -> N
CB -> H
NH -> C
HB -> C
HC -> B
HN -> C
NN -> C
BH -> H
NC -> B
NB -> B
BN -> B
BB -> N
BC -> B
CC -> N
CN -> C
 
 The first line is the polymer template - this is the starting point of the process. 
 The following section defines the pair insertion rules. A rule like AB -> C means that when elements A and B are immediately adjacent, element C should be inserted between them. These insertions all happen simultaneously. 
 So, starting with the polymer template NNCB , the first step simultaneously considers all three pairs: 
 
 The first pair ( NN ) matches the rule NN -> C , so element C is inserted between the first N and the second N . 
 The second pair ( NC ) matches the rule NC -> B , so element B is inserted between the N and the C . 
 The third pair ( CB ) matches the rule CB -> H , so element H is inserted between the C and the B . 
 
 Note that these pairs overlap: the second element of one pair is the first element of the next pair. Also, because all pairs are considered simultaneously, inserted elements are not considered to be part of a pair until the next step. 
 After the first step of this process, the polymer becomes N C N B C H B . 
 Here are the results of a few steps using the above rules: 
 Template: NNCB
After step 1: NCNBCHB
After step 2: NBCCNBBBCBHCB
After step 3: NBBBCNCCNBBNBNBBCHBHHBCHB
After step 4: NBBNBNBBCCNBCNCCNBBNBBNBBBNBBNBBCBHCBHHNHCBBCBHCB
 
 This polymer grows quickly. After step 5, it has length 97; After step 10, it has length 3073. After step 10, B occurs 1749 times, C occurs 298 times, H occurs 161 times, and N occurs 865 times; taking the quantity of the most common element ( B , 1749) and subtracting the quantity of the least common element ( H , 161) produces 1749 - 161 = 1588 
 . 
 Apply 10 steps of pair insertion to the polymer template and find the most and least common elements in the result. What do you get if you take the quantity of the most common element and subtract the quantity of the least common element?

(cached)

--- Part Two --- The resulting polymer isn't nearly strong enough to reinforce the submarine. You'll need to run more steps of the pair insertion process; a total of 40 steps should do it. 
 In the above example, the most common element is B (occurring 2192039569602 times) and the least common element is H (occurring 3849876073 times); subtracting these produces 2188189693529 . 
 Apply 40 steps of pair insertion to the polymer template and find the most and least common elements in the result. What do you get if you take the quantity of the most common element and subtract the quantity of the least common element?

(cached)

## Input
```
PSVVKKCNBPNBBHNSFKBO
...
```

### Solve today

`aoc::solve input body`:
    The body is executed as a coroutine. Input is available as the `$input` variable. The results are returned using `[yield]`

## Scalable approach

A brute force approach will not work for large `n`. The approach is to track:

1. The amount of pairs to apply the rules `pairs`
2. Update the pairs and the total count of elements `counts` based on the production rules.

In [5]:
set test {NNCB

CH -> B
HH -> N
CB -> H
NH -> C
HB -> C
HC -> B
HN -> C
NN -> C
BH -> H
NC -> B
NB -> B
BN -> B
BB -> N
BC -> B
CC -> N
CN -> C}

proc step {pairs rules} {
    upvar counts counts
    set newpairs $pairs
    foreach {pair count} $pairs {
#         puts $pair
        set rule [dict get $rules $pair]
#          puts $pair->$rule
         incr counts($rule) $count
        set produced [join [lmap r $rule {list [string index $pair 0]$rule $rule[string index $pair 1] }  ] ""]
        foreach product $produced {
#             puts $product
            dict incr newpairs $product $count
        }
        dict incr newpairs $pair -$count
    }
    return $newpairs
}

proc solve {input steps} {
    lassign [split [string map [list \n\n @] $input] @] template rules
    set template [split $template {}]
    foreach t $template {
        incr counts($t)
    }
    set rules [join [lmap r [split $rules \n] {lassign $r from _ to ; list $from $to}]]
    set pairs {}
    foreach left [lrange $template 0 end-1] right [lrange $template 1 end] {
        dict incr pairs $left$right
    }
    set pairs
#     parray counts


    time {set pairs [step $pairs $rules]} $steps
    set total [lsort -stride 2 -index 1 -dictionary [array get counts]]
    expr {[lindex $total end] - [lindex $total 1]}
}
# array get counts

In [6]:
aoc::solve $test {
    # $input is available in the body
    # return the results using yield

    yield [solve $input 10]
    yield [solve $input 40]
} 

Part1	1588 (1574 us)
Part2	2188189693529 (4057 us)


In [11]:
aoc::solve $input {
    # $input is available in the body
    # return the results using yield

    yield [solve $input 10]
    yield [solve $input 40]
} 

Part1	2584 (4428 us)
Part2	3816397135460 (14079 us)
